In [1]:
%pip install TA-Lib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import pandas as pd
import numpy as np
import talib

In [3]:
def get_files(input_dir: str):
    files = os.listdir(input_dir)
    files = [os.path.join(input_dir, file) for file in files if file.endswith(".parquet")]
    return files

In [4]:
def safe_read_parquet(path: str) -> pd.DataFrame:
    try:
        return pd.read_parquet(path, engine="pyarrow")
    except Exception:
        import pyarrow.parquet as pq
        table = pq.read_table(path, arrow_extensions_enabled=False)
        return table.to_pandas()

In [5]:
def rsi_calculator(df: pd.DataFrame) -> pd.DataFrame:
    fd_col = next((c for c in df.columns if c.startswith("fd_close_d")), None)
    if fd_col is None:
        raise ValueError("No fractional-differentiated close column found (expected prefix 'fd_close_d').")

    data = pd.to_numeric(df[fd_col], errors="coerce").to_numpy(dtype=np.float64)
    df["RSI_14"] = talib.RSI(data, timeperiod=14)
    return df

In [6]:
def get_raw_path(file_path: str) -> str:
    return file_path.replace("dollar_bars_fracdiff_parquet", "dollar_bars_parquet")

In [7]:
def get_t_events(close: pd.Series, h: float) -> pd.DatetimeIndex:
    if h is None or not np.isfinite(h) or h <= 0:
        raise ValueError("CUSUM threshold h must be a positive finite float.")

    close = pd.to_numeric(close, errors="coerce")
    close.index = pd.DatetimeIndex(close.index)
    close = close.dropna()
    if close.index.has_duplicates:
        close = close.groupby(level=0).last().sort_index()

    diff = np.log(close).diff().dropna()

    s_pos, s_neg = 0.0, 0.0
    t_events = []
    for t, change in diff.items():
        s_pos = max(0.0, s_pos + float(change))
        s_neg = min(0.0, s_neg + float(change))
        if s_pos > h:
            s_pos = 0.0
            t_events.append(t)
        elif s_neg < -h:
            s_neg = 0.0
            t_events.append(t)

    return pd.DatetimeIndex(t_events)

def add_vertical_barrier(t_events: pd.DatetimeIndex, close_index: pd.DatetimeIndex, horizon_bars: int = 5) -> pd.Series:
    if horizon_bars <= 0:
        raise ValueError("horizon_bars must be >= 1")
    if len(t_events) == 0:
        return pd.Series(dtype="datetime64[ns]")

    close_index = pd.DatetimeIndex(close_index).sort_values().drop_duplicates()
    t_events = pd.DatetimeIndex(t_events).sort_values()

    t_pos = close_index.searchsorted(t_events)
    t1_pos = t_pos + int(horizon_bars)
    valid = t1_pos < len(close_index)

    t1 = pd.Series(close_index[t1_pos[valid]], index=t_events[valid])
    return t1

def mp_pandas_obj(func, pd_obj, num_threads: int = 4, **kwargs):
    from concurrent.futures import ThreadPoolExecutor

    molecules = [m for m in np.array_split(pd_obj[1], max(1, int(num_threads))) if len(m) > 0]
    if not molecules:
        return pd.DataFrame()

    with ThreadPoolExecutor(max_workers=max(1, int(num_threads))) as executor:
        futures = [executor.submit(func, molecule=molecule, **kwargs) for molecule in molecules]
        out = [f.result() for f in futures]

    if isinstance(out[0], pd.DataFrame):
        return pd.concat(out).sort_index()
    if isinstance(out[0], pd.Series):
        return pd.concat(out).sort_index()
    return out

def apply_pt_sl_on_t1(close: pd.Series, events: pd.DataFrame, pt_sl: tuple, molecule) -> pd.DataFrame:
    close = pd.to_numeric(close, errors="coerce")
    close.index = pd.DatetimeIndex(close.index)
    close = close.dropna()
    if close.index.has_duplicates:
        close = close.groupby(level=0).last().sort_index()

    events_ = events.loc[molecule]
    out = pd.DataFrame(index=events_.index, columns=["t1", "pt", "sl"], dtype="datetime64[ns]")
    out["t1"] = events_["t1"]

    pt_mult, sl_mult = float(pt_sl[0]), float(pt_sl[1])

    for loc, vertical_t1 in events_["t1"].items():
        if pd.isna(vertical_t1):
            continue

        path = close.loc[loc:vertical_t1]
        if path.empty:
            continue

        ret = path / float(path.iloc[0]) - 1.0
        if "side" in events_.columns:
            ret = ret * float(events_.at[loc, "side"] )

        trgt = float(events_.at[loc, "trgt"] )
        pt_level = pt_mult * trgt if pt_mult > 0 else np.inf
        sl_level = -sl_mult * trgt if sl_mult > 0 else -np.inf

        pt_touch = ret[ret > pt_level]
        sl_touch = ret[ret < sl_level]

        out.at[loc, "pt"] = pt_touch.index.min() if not pt_touch.empty else pd.NaT
        out.at[loc, "sl"] = sl_touch.index.min() if not sl_touch.empty else pd.NaT

    return out

def get_events(
    close: pd.Series,
    t_events: pd.DatetimeIndex,
    pt_sl: tuple,
    target: pd.Series,
    min_ret: float,
    num_threads: int = 4,
    vertical_horizon_bars: int = 5,
    side: pd.Series = None,
) -> pd.DataFrame:
    close = pd.to_numeric(close, errors="coerce")
    close.index = pd.DatetimeIndex(close.index)
    close = close.dropna()
    if close.index.has_duplicates:
        close = close.groupby(level=0).last().sort_index()

    target = pd.to_numeric(target, errors="coerce")
    target.index = pd.DatetimeIndex(target.index)
    target = target.dropna()
    if target.index.has_duplicates:
        target = target.groupby(level=0).last().sort_index()

    trgt = target.reindex(t_events)
    trgt = trgt[trgt > float(min_ret)].dropna()
    if trgt.empty:
        return pd.DataFrame(columns=["t1", "trgt"])

    t1 = add_vertical_barrier(trgt.index, close.index, horizon_bars=vertical_horizon_bars)
    events = pd.DataFrame(index=trgt.index)
    events["t1"] = t1
    events["trgt"] = trgt

    if side is not None:
        side_ = side.reindex(events.index).dropna()
        events = events.loc[side_.index]
        events["side"] = side_

    first_touch = mp_pandas_obj(
        apply_pt_sl_on_t1,
        pd_obj=("molecule", events.index),
        num_threads=num_threads,
        close=close,
        events=events,
        pt_sl=pt_sl,
    )

    events["t1"] = first_touch[["t1", "pt", "sl"]].min(axis=1)
    return events.dropna(subset=["t1", "trgt"])

def get_bins(events: pd.DataFrame, close: pd.Series) -> pd.DataFrame:
    if events.empty:
        return pd.DataFrame(columns=["ret", "bin"])

    close = pd.to_numeric(close, errors="coerce")
    close.index = pd.DatetimeIndex(close.index)
    close = close.dropna()
    if close.index.has_duplicates:
        close = close.groupby(level=0).last().sort_index()

    events_ = events.dropna(subset=["t1"]).copy()
    if events_.empty:
        return pd.DataFrame(columns=["ret", "bin"])

    events_ = events_[events_.index.isin(close.index)]
    events_ = events_[pd.DatetimeIndex(events_["t1"]).isin(close.index)]
    if events_.empty:
        return pd.DataFrame(columns=["ret", "bin"])

    entry_px = close.loc[events_.index].to_numpy()
    exit_px = close.loc[pd.DatetimeIndex(events_["t1"])].to_numpy()

    out = pd.DataFrame(index=events_.index)
    out["ret"] = exit_px / entry_px - 1.0

    if "side" in events_.columns:
        out["ret"] = out["ret"] * events_["side"].to_numpy()
        out["bin"] = (out["ret"] > 0).astype(float)
    else:
        out["bin"] = np.sign(out["ret"] )

    out.loc[out["ret"] == 0, "bin"] = 0.0
    return out

def get_sample_weight_uniqueness(events: pd.DataFrame, close_index: pd.DatetimeIndex) -> pd.Series:
    if events.empty:
        return pd.Series(dtype=float)

    index = pd.DatetimeIndex(close_index).sort_values().drop_duplicates()
    events_ = events.dropna(subset=["t1"]).copy()
    if events_.empty:
        return pd.Series(dtype=float)

    starts = index.searchsorted(pd.DatetimeIndex(events_.index), side="left")
    ends = index.searchsorted(pd.DatetimeIndex(events_["t1"]), side="left")

    valid = (starts >= 0) & (starts < len(index)) & (ends >= starts) & (ends < len(index))
    starts = starts[valid]
    ends = ends[valid]
    evt_idx = events_.index[valid]

    if len(evt_idx) == 0:
        return pd.Series(dtype=float)

    diff = np.zeros(len(index) + 1, dtype=np.int64)
    np.add.at(diff, starts, 1)
    np.add.at(diff, ends + 1, -1)
    concurrency = np.cumsum(diff[:-1])

    inv_conc = np.zeros_like(concurrency, dtype=np.float64)
    pos = concurrency > 0
    inv_conc[pos] = 1.0 / concurrency[pos]
    prefix = np.cumsum(inv_conc)

    sums = prefix[ends] - np.where(starts > 0, prefix[starts - 1], 0.0)
    lengths = (ends - starts + 1).astype(np.float64)
    uniqueness = sums / lengths

    w = pd.Series(uniqueness, index=evt_idx, dtype=np.float64)
    total = w.sum()
    if np.isfinite(total) and total > 0:
        w = w * (len(w) / total)
    return w

In [8]:
def mp_pandas_obj(func, pd_obj, num_threads: int = 4, **kwargs):
    from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
    import os

    molecules = [m for m in np.array_split(pd_obj[1], max(1, int(num_threads))) if len(m) > 0]
    if not molecules:
        return pd.DataFrame()

    max_workers = max(1, min(int(num_threads), os.cpu_count() or 1))

    if max_workers == 1:
        out = [func(molecule=m, **kwargs) for m in molecules]
    else:
        try:
            with ProcessPoolExecutor(max_workers=max_workers) as executor:
                futures = [executor.submit(func, molecule=molecule, **kwargs) for molecule in molecules]
                out = [f.result() for f in futures]
        except Exception:
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                futures = [executor.submit(func, molecule=molecule, **kwargs) for molecule in molecules]
                out = [f.result() for f in futures]

    if isinstance(out[0], pd.DataFrame):
        return pd.concat(out).sort_index()
    if isinstance(out[0], pd.Series):
        return pd.concat(out).sort_index()
    return out

In [9]:
def calculate_rolling_volitility_and_barrier(
    df: pd.DataFrame,
    vol_span: int = 250,
    pt_width: float = 2.0,
    sl_width: float = 2.0,
    horizon_bars: int = 5,
    ts_col: str = "end_ts",
    side: pd.Series = None,
    min_ret: float = 1e-5,
    num_threads: int = 4,
    cusum_h: float = None,
 ) -> pd.DataFrame:
    out = df.copy()

    if ts_col not in out.columns:
        ts_col = "start_ts" if "start_ts" in out.columns else None
    if ts_col is None:
        raise ValueError("Expected 'end_ts' or 'start_ts' in raw dataframe for event labeling.")

    out[ts_col] = pd.to_datetime(out[ts_col], errors="coerce")
    out = out.dropna(subset=[ts_col]).sort_values(ts_col).reset_index(drop=True)

    close = pd.to_numeric(out["close"], errors="coerce")
    close.index = pd.DatetimeIndex(out[ts_col])
    close = close.dropna()
    if close.index.has_duplicates:
        close = close.groupby(level=0).last().sort_index()
    else:
        close = close.sort_index()

    returns = close.pct_change()
    trgt = returns.ewm(span=vol_span, adjust=False).std()

    if cusum_h is None:
        h = float(trgt.dropna().median())
        if not np.isfinite(h) or h <= 0:
            h = float(trgt.dropna().mean())
    else:
        h = float(cusum_h)

    if not np.isfinite(h) or h <= 0:
        return pd.DataFrame(columns=["t1", "rolling_vol", "ret", "Target_Y", "sample_weight"] )

    t_events = get_t_events(close, h=h)
    events = get_events(
        close=close,
        t_events=t_events,
        pt_sl=(pt_width, sl_width),
        target=trgt,
        min_ret=min_ret,
        num_threads=num_threads,
        vertical_horizon_bars=horizon_bars,
        side=side,
    )

    bins = get_bins(events, close)
    sample_weight = get_sample_weight_uniqueness(events, close.index)

    labeled = events.join(bins[["ret", "bin"]], how="left")
    labeled = labeled.join(sample_weight.rename("sample_weight"), how="left")
    labeled = labeled.rename(columns={"trgt": "rolling_vol", "bin": "Target_Y"})
    labeled.index.name = ts_col
    return labeled

In [10]:
def process_single_file(file_path: str, output_dir: str):
    df = safe_read_parquet(file_path)
    df_rsi = rsi_calculator(df)

    fd_col = next((c for c in df_rsi.columns if c.startswith("fd_close_d")), None)
    if fd_col is None:
        raise ValueError(f"No fractional close column found in {file_path}")

    raw_path = get_raw_path(file_path)
    raw_df = safe_read_parquet(raw_path)
    result_df = calculate_rolling_volitility_and_barrier(
        raw_df,
        vol_span=250,
        horizon_bars=5,
        pt_width=2.0,
        sl_width=2.0,
        min_ret=1e-5,
        num_threads=1,
    )

    merge_key = "end_ts"
    if merge_key not in df_rsi.columns:
        merge_key = "start_ts"
    if merge_key not in df_rsi.columns:
        raise ValueError(f"Missing timestamp merge key in {file_path}; expected end_ts or start_ts.")

    feature_df = df_rsi[[merge_key, fd_col, "RSI_14"]].copy()
    feature_df[merge_key] = pd.to_datetime(feature_df[merge_key], errors="coerce")
    feature_df = feature_df.rename(columns={fd_col: "fd_close"})

    label_df = result_df.reset_index().rename(columns={result_df.index.name: merge_key})
    label_df[merge_key] = pd.to_datetime(label_df[merge_key], errors="coerce")

    final_ml_dataset = feature_df.merge(
        label_df[[merge_key, "t1", "rolling_vol", "Target_Y", "sample_weight"]],
        on=merge_key,
        how="inner",
    )
    final_ml_dataset = final_ml_dataset.sort_values(merge_key).reset_index(drop=True)

    final_ml_dataset = final_ml_dataset.dropna(
        subset=["fd_close", "RSI_14", "rolling_vol", "Target_Y", "sample_weight"]
    )

    out_path = os.path.join(output_dir, os.path.basename(file_path))
    final_ml_dataset.to_parquet(out_path, index=False)
    return out_path

In [11]:
input_dir = "./dollar_bars_fracdiff_parquet"
output_dir = "./training_data"
os.makedirs(output_dir, exist_ok=True)

files = get_files(input_dir)

def run_parallel_files(max_workers=None):
    from concurrent.futures import ThreadPoolExecutor, as_completed

    cpu_total = os.cpu_count() or 1
    if max_workers is None:
        max_workers = min(12, max(1, cpu_total - 1))
    max_workers = max(1, int(max_workers))

    if len(files) == 0:
        print("No files found in input_dir.")
        return

    print(f"Processing {len(files)} files with ThreadPoolExecutor ({max_workers} workers)...")

    out_paths = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_single_file, file_path, output_dir): file_path for file_path in files}
        for fut in as_completed(futures):
            file_path = futures[fut]
            try:
                out_path = fut.result()
                out_paths.append(out_path)
                print(f"Saved: {out_path}")
            except Exception as exc:
                print(f"ERROR: {file_path} -> {exc}")

    print(f"Completed {len(out_paths)}/{len(files)} files.")

run_parallel_files()

Processing 826 files with ThreadPoolExecutor (12 workers)...
Saved: ./training_data\ABVX.parquet
Saved: ./training_data\ACGL.parquet
Saved: ./training_data\A.parquet
Saved: ./training_data\ABT.parquet
Saved: ./training_data\ACI.parquet
Saved: ./training_data\ABNB.parquet
Saved: ./training_data\ABBV.parquet
Saved: ./training_data\AAL.parquet
Saved: ./training_data\AA.parquet
Saved: ./training_data\AAPL.parquet
Saved: ./training_data\ACHR.parquet
Saved: ./training_data\AAOI.parquet
Saved: ./training_data\ACN.parquet
Saved: ./training_data\ACWI.parquet
Saved: ./training_data\ADBE.parquet
Saved: ./training_data\ADM.parquet
Saved: ./training_data\ACWX.parquet
Saved: ./training_data\ADI.parquet
Saved: ./training_data\AEE.parquet
Saved: ./training_data\AEM.parquet
Saved: ./training_data\AEIS.parquet
Saved: ./training_data\ADSK.parquet
Saved: ./training_data\AEP.parquet
Saved: ./training_data\AEO.parquet
Saved: ./training_data\ADP.parquet
Saved: ./training_data\AFL.parquet
Saved: ./training_da

In [13]:
# Smoke test: run one file and validate output for ML handoff
required_cols = ["fd_close", "RSI_14", "t1", "rolling_vol", "Target_Y", "sample_weight"]

if len(files) == 0:
    raise ValueError("No files found in input_dir.")

test_file = files[0]
print(f"Testing file: {os.path.basename(test_file)}")

out_path = process_single_file(test_file, output_dir)
print(f"Output: {out_path}")

test_df = safe_read_parquet(out_path)
print(f"Rows: {len(test_df):,}")
print(f"Columns: {list(test_df.columns)}")

missing_cols = [c for c in required_cols if c not in test_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

check_df = test_df[required_cols].copy()
nan_counts = check_df.isna().sum()
print("NaN counts in required columns:")
print(nan_counts.to_string())

weights = pd.to_numeric(test_df["sample_weight"], errors="coerce")
if not np.isfinite(weights).all():
    raise ValueError("sample_weight contains non-finite values.")
if (weights < 0).any():
    raise ValueError("sample_weight contains negative values.")

print(f"sample_weight mean: {weights.mean():.6f}")
print(f"sample_weight min/max: {weights.min():.6f} / {weights.max():.6f}")
print("Target_Y distribution:")
print(test_df["Target_Y"].value_counts(dropna=False).sort_index().to_string())

print("Smoke test passed.")

Testing file: A.parquet
Output: ./training_data\A.parquet
Rows: 17,489
Columns: ['end_ts', 'fd_close', 'RSI_14', 't1', 'rolling_vol', 'Target_Y', 'sample_weight']
NaN counts in required columns:
fd_close         0
RSI_14           0
t1               0
rolling_vol      0
Target_Y         0
sample_weight    0
sample_weight mean: 1.009281
sample_weight min/max: 0.299932 / 1.799594
Target_Y distribution:
Target_Y
-1.0    8745
 0.0      68
 1.0    8676
Smoke test passed.
